# EYE-D Server API Verification

이 노트북은 서버의 핵심 기능(DB 연결, Re-ID 매칭, 데이터 저장)이 정상적으로 동작하는지 검증합니다.

## 0. 서버 실행 (Background)
서버가 아직 실행 중이지 않다면 아래 셀을 실행하여 백그라운드에서 서버를 구동합니다.

In [2]:
import subprocess
import time
import os
import requests

# 현재 디렉토리 확인 및 server 폴더로 이동 준비
cwd = os.getcwd()
if "notebooks" in cwd:
    server_root = os.path.abspath(os.path.join(cwd, ".."))
else:
    server_root = cwd

print(f"Starting server and DB at: {server_root}")

# 1. DB 실행 (Docker Compose)
print("Starting database...")
subprocess.run(["docker", "compose", "up", "-d"], cwd=server_root)
time.sleep(3) # DB 준비 대기

# 2. 기존 uvicorn 프로세스 종료 (포트 충돌 방지)
subprocess.run(["pkill", "-f", "uvicorn"])
time.sleep(1)

# 3. uvicorn 실행 (stdout/stderr는 server_log.txt에 기록)
with open(os.path.join(server_root, "server_log.txt"), "w") as f:
    proc = subprocess.Popen(
        ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
        cwd=server_root,
        stdout=f,
        stderr=f
    )

print(f"Server started with PID: {proc.pid}")
print("Waiting 3 seconds for server to stabilize...")
time.sleep(3)

Starting server and DB at: /home/torious/projects/tmp/EYE-D-for-UI/EYE-D/server
Starting database...


 Network server_default Creating 
 Network server_default Created 
 Container kote-postgres Creating 
 Container kote-postgres Created 
 Container kote-postgres Starting 
 Container kote-postgres Started 


Server started with PID: 97496
Waiting 3 seconds for server to stabilize...


## 1. Health Check
서버와 PostgreSQL DB가 연결되어 있는지 확인합니다.

In [3]:
BASE_URL = "http://127.0.0.1:8000"
DETECTION_API = f"{BASE_URL}/api/v1/security/detections"

try:
    response = requests.get(f"{BASE_URL}/health")
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"Error connecting to server: {e}")

Status: 200
Response: {'status': 'ok', 'db': 1}


## 2. Detection Data Ingestion (New Person)
새로운 인물의 특징 벡터(OSNet 임베딩)를 전송합니다.

In [4]:
import random
import json
from datetime import datetime

def generate_mock_detection(camera_id="CAM_01", event_type="detection", embedding=None):
    if embedding is None:
        # 512차원 랜덤 벡터 생성
        embedding = [random.uniform(-0.1, 0.1) for _ in range(512)]
    
    return {
        "camera_id": camera_id,
        "tracklet_id": f"track_{random.randint(1000, 9999)}",
        "embedding_identity": embedding,
        "timestamp": datetime.now().isoformat(),
        "bbox": [100.0, 150.0, 200.0, 400.0],
        "event_type": event_type
    }

# 1. 새로운 인물 데이터 발송
new_person_data = generate_mock_detection()
res1 = requests.post(DETECTION_API, json=new_person_data)
result1 = res1.json()

print(f"New Person Result: {json.dumps(result1, indent=2)}")
target_global_id = result1['global_id']
saved_embedding = new_person_data['embedding_identity']

New Person Result: {
  "detection_id": 1,
  "global_id": 1,
  "matched": false,
  "similarity": null
}


## 3. Re-ID Matching (Same Person)
이전에 저장한 벡터와 유사한 벡터를 보내 동일 인물로 매칭되는지 확인합니다.

In [5]:
# 기존 벡터에 약간의 노이즈 추가 (매우 유사한 벡터 생성)
similar_embedding = [v + random.uniform(-0.01, 0.01) for v in saved_embedding]

match_data = generate_mock_detection(embedding=similar_embedding)
res2 = requests.post(DETECTION_API, json=match_data)
result2 = res2.json()

print(f"Re-ID Result: {json.dumps(result2, indent=2)}")
if result2['global_id'] == target_global_id:
    print("✅ Success: Re-ID matched correctly!")
else:
    print("❌ Failure: Re-ID failed to match.")

Re-ID Result: {
  "detection_id": 2,
  "global_id": 1,
  "matched": true,
  "similarity": 0.9951422230786221
}
✅ Success: Re-ID matched correctly!


## 4. Track History Retrieval
방금 저장된 인물의 이동 경로를 조회합니다.

In [6]:
track_url = f"{BASE_URL}/api/v1/security/persons/{target_global_id}/track"
res3 = requests.get(track_url)
tracks = res3.json()

print(f"Found {len(tracks)} tracks for global_id {target_global_id}")
for t in tracks:
    print(f"- Time: {t['detected_at']}, Cam: {t['camera_id']}, BBox: {t['bbox']}")

Found 2 tracks for global_id 1
- Time: 2026-05-15T06:39:11.201627+00:00, Cam: CAM_01, BBox: [100.0, 150.0, 200.0, 400.0]
- Time: 2026-05-15T06:39:15.343155+00:00, Cam: CAM_01, BBox: [100.0, 150.0, 200.0, 400.0]


## 5. 서버 및 DB 종료
테스트가 완료되면 아래 셀을 실행하여 서버와 DB를 안전하게 종료합니다.

In [7]:
print("Stopping server...")
subprocess.run(["pkill", "-f", "uvicorn"])
print("Server stopped.")

print("Stopping database...")
subprocess.run(["docker", "compose", "down"], cwd=server_root)
print("Database stopped.")

Stopping server...
Server stopped.
Stopping database...


 Container kote-postgres Stopping 
 Container kote-postgres Stopped 
 Container kote-postgres Removing 
 Container kote-postgres Removed 
 Network server_default Removing 


Database stopped.


 Network server_default Removed 


## 6. Shell (cURL) Equivalent Commands
위에서 Python `requests` 모듈을 이용해 수행한 작업을 터미널(Shell)에서 `curl` 명령어로 동일하게 테스트하려면 아래 명령어들을 사용할 수 있습니다.

### 1) 서버 및 DB 구동
```bash
cd server
docker compose up -d
uvicorn app.main:app --host 127.0.0.1 --port 8000
```

### 2) Health Check
```bash
curl -X GET "http://127.0.0.1:8000/health"
```

### 3) Detection Data Ingestion (New Person)
```bash
curl -X POST "http://127.0.0.1:8000/api/v1/security/detections" \
     -H "Content-Type: application/json" \
     -d '{
           "camera_id": "CAM_01",
           "tracklet_id": "track_1234",
           "embedding_identity": [0.05, -0.02, 0.01], 
           "timestamp": "2026-05-15T15:30:00.000Z",
           "bbox": [100.0, 150.0, 200.0, 400.0],
           "event_type": "detection"
         }'
```
*(Note: `embedding_identity`는 실제로는 모델이 생성한 512차원 등의 벡터여야 합니다)*

### 4) Re-ID Matching
위와 유사한 벡터(동일 인물)를 새로운 `tracklet_id`와 타임스탬프로 전송하여 매칭되는지 확인합니다.
```bash
curl -X POST "http://127.0.0.1:8000/api/v1/security/detections" \
     -H "Content-Type: application/json" \
     -d '{
           "camera_id": "CAM_01",
           "tracklet_id": "track_5678",
           "embedding_identity": [0.051, -0.019, 0.012],
           "timestamp": "2026-05-15T15:30:10.000Z",
           "bbox": [100.0, 150.0, 200.0, 400.0],
           "event_type": "detection"
         }'
```

### 5) Track History Retrieval
위 결과에서 반환된 `global_id` (예: 10)를 사용해 추적 이력을 조회합니다.
```bash
curl -X GET "http://127.0.0.1:8000/api/v1/security/persons/10/track"
```

### 6) 서버 및 DB 종료
```bash
pkill -f uvicorn
cd server
docker compose down
```